In [ ]:
import os, json, warnings
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from Learn.features import donchian_trend
warnings.filterwarnings('ignore')

## Helpers

In [ ]:
def load_ohlcv(ds_name, start_date=None, n_rows=None):
    """Load an OHLCV CSV, optionally sliced by start_date or tail n_rows."""
    df = pd.read_csv(ds_name)
    df = df.sort_values('Time').reset_index(drop=True)
    df['Time'] = pd.to_datetime(df['Time'])
    if start_date:
        df = df[df['Time'] > start_date].reset_index(drop=True)
    if n_rows is not None:
        df = df.tail(n_rows).reset_index(drop=True)
    return df

## Configuration

Set the dataset path and evaluation window. `TEST_DATE` trims the dataframe to only post-date rows so the chart isn't overwhelmed — set to `None` to use the full file.

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
DS_NAME    = '../data/US500_M1_520weeks.csv'
TEST_DATE  = '2026-04-01'   # Show bars after this date only (None = full file)

# ── Load ──────────────────────────────────────────────────────────────────────
df = load_ohlcv(DS_NAME, start_date=TEST_DATE)

print(f"Dataset: {DS_NAME.split('/')[-1]}")
print(f"Window:  {df['Time'].iloc[0]}  →  {df['Time'].iloc[-1]}")
print(f"Bars:    {len(df):,}")

## Label Parameters

Tune the regime-detection and triple-barrier labelling parameters below, then run **Compute Labels** to see updated signals on the chart.

| Group | Key Parameters |
|---|---|
| **Regime** | `ma_period`, `slope_threshold`, `atr_percentile` — controls trend detection sensitivity |
| **Labelling** | `tp_mult`, `sl_mult`, `max_horizon` — risk-reward and time limit for each label |
| **Selectivity** | `trend_pullback_thresh` — raise to label fewer, higher-quality entries; `skip_range` masks range-bound bars |

Use the **Save / Load Parameters** cell below to persist a named config to `label_params.json`.

In [ ]:
# ── Instrument / Timeframe identifier (used as key in label_params.json) ──────
PARAM_KEY = DS_NAME.split('/')[-1].split('_')[0]+'_M1_prod'

# ── Regime detection ──────────────────────────────────────────────────────────
regime_params = {
      "ma_period": 90,
      "slope_smoothness": 50,
      "regime_min_duration": 0,
      "atr_window": 60,
      "atr_lookback": 720,
      "atr_percentile": 0.0,
      "slope_threshold": 0 # 1e-8
    }

# ── Triple-barrier labelling ──────────────────────────────────────────────────
label_params = {
      "z_window": 14,
      "z_thresh": 1,
      "z_limit": 5,
      "atr_window": 14,
      "tp_mult": 2.5,
      "sl_mult": 2.5,
      "max_horizon": 30,
      "trend_pullback_thresh": -1.0,
      "regime_params": regime_params,
      "skip_range": True
    }

# ── Outcome params (subset used for P&L simulation) ──────────────────────────
outcome_params = {k: label_params[k] for k in ['atr_window', 'tp_mult', 'sl_mult', 'max_horizon']}

print(f"Active config:  '{PARAM_KEY}'")

In [ ]:
## Compute Labels

# Runs the causal triple-barrier labeller, the trade-outcome calculator, and the regime detector. Re-run this cell after changing any parameter above.
from Learn.labels import (
    causal_triple_barrier_hilow_trend_labeler,
    causal_market_regime,
    super_smoother,
    MFE_filter_outcomes
)

from talib import EMA, ATR

df, df_outcomes = MFE_filter_outcomes(df, mfe_thresh=1, mfa_thresh=0.30, label_params=label_params, outcome_params=outcome_params)

In [ ]:
# ── Regime & slope ────────────────────────────────────────────────────────────
df['Centered_MA']   = EMA(df['Close'], timeperiod=regime_params['ma_period'])
df['Regime']        = causal_market_regime(df, **regime_params)
slope               = df['Centered_MA'].diff()
df['slope_sm']      = super_smoother(slope, regime_params['slope_smoothness'])
df['donchian_trend'] = donchian_trend(df, length=regime_params['atr_window'])

regime_counts = df['Regime'].value_counts().rename({1: 'Up', 0: 'Range', -1: 'Down'})
print(f"Regime   — {dict(regime_counts)}")
print("Label distribution:")
print(df['target'].value_counts(normalize=True).rename({-1: 'SELL', 0: 'HOLD', 1: 'BUY'})*100)

In [ ]:
# # ── Trade outcomes ────────────────────────────────────────────────────────────
# df_outcomes = calculate_trade_outcomes_all_candles(df, **outcome_params)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df_plot = df.copy().tail(5000).reset_index(drop=True)

buy_bars  = df_plot[df_plot['target'] ==  1]
sell_bars = df_plot[df_plot['target'] == -1]

# Calculate ATR
df_plot['ATR'] = ATR(df_plot['High'], df_plot['Low'], df_plot['Close'], timeperiod=label_params['atr_window'])

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.7, 0.3],
    vertical_spacing=0.02,
)

# ── Candlesticks ──────────────────────────────────────────────────────────────
fig.add_trace(go.Candlestick(
    x=df_plot['Time'],
    open=df_plot['Open'], high=df_plot['High'],
    low=df_plot['Low'],   close=df_plot['Close'],
    name='OHLCV',
    increasing_line_color='#26a69a', decreasing_line_color='#ef5350',
    increasing_fillcolor='#26a69a',  decreasing_fillcolor='#ef5350',
    line=dict(width=1),
    hoverinfo='skip',
), row=1, col=1)

# ── ATR Bands (Upper & Lower) ─────────────────────────────────────────────────
atr_mult = label_params['tp_mult']
df_plot['Upper_Band'] = df_plot['Close'] + df_plot['ATR'] * atr_mult
df_plot['Lower_Band'] = df_plot['Close'] - df_plot['ATR'] * atr_mult

fig.add_trace(go.Scatter(
    x=df_plot['Time'], y=df_plot['Upper_Band'],
    mode='lines',
    line=dict(width=1, color='rgba(0, 230, 118, 0.3)', dash='dot'),
    name=f'TP Upper ({atr_mult}×ATR)',
    hoverinfo='skip',
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=df_plot['Time'], y=df_plot['Lower_Band'],
    mode='lines',
    line=dict(width=1, color='rgba(255, 23, 68, 0.3)', dash='dot'),
    name=f'TP Lower ({atr_mult}×ATR)',
    hoverinfo='skip',
), row=1, col=1)

# ── BUY markers ───────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=buy_bars['Time'], y=buy_bars['Low'],
    mode='markers',
    marker=dict(symbol='triangle-up', size=14, color='#00e676',
                line=dict(color='darkgreen', width=0.8)),
    name='BUY label',
    hoverinfo='skip',
), row=1, col=1)

# ── SELL markers ──────────────────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=sell_bars['Time'], y=sell_bars['High'],
    mode='markers',
    marker=dict(symbol='triangle-down', size=14, color='#ff1744',
                line=dict(color='darkred', width=0.8)),
    name='SELL label',
    hoverinfo='skip',
), row=1, col=1)

# ── Regime-coloured MA ────────────────────────────────────────────────────────
for regime_val, color, label in [(1, '#26a69a', 'Uptrend'), (0, '#888888', 'Range'), (-1, '#ef5350', 'Downtrend')]:
    fig.add_trace(go.Scatter(
        x=df_plot['Time'],
        y=df_plot['Centered_MA'].where(df_plot['Regime'] == regime_val),
        mode='lines', line=dict(width=2, color=color), name=label,
        hoverinfo='skip',
    ), row=1, col=1)

# ── Slope & Donchian panels ───────────────────────────────────────────────────
fig.add_trace(go.Scatter(
    x=df_plot['Time'], y=df_plot['slope_sm'],
    mode='lines', line=dict(width=1, color='#82aaff'), name='Smoothed Slope',
    hoverinfo='skip',
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=df_plot['Time'], y=df_plot['donchian_trend'],
    mode='lines', line=dict(width=1, color='#ffcb6b'), name='Donchian Trend',
    hoverinfo='skip',
), row=2, col=1)

# ── Layout ────────────────────────────────────────────────────────────────────
title_str = (
    f"BUY: {len(buy_bars):,}   SELL: {len(sell_bars):,}   "
    f"({(len(buy_bars) + len(sell_bars)) / len(df_plot) * 100:.2f}% active)  |  "
)

fig.update_layout(
    title=dict(text=title_str, font_size=13),
    xaxis_rangeslider_visible=False,
    xaxis2=dict(title='Time'),
    yaxis=dict(title='Price', fixedrange=False),
    yaxis2=dict(title='Indicator'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=1000,
    margin=dict(l=60, r=20, t=80, b=60),
    hovermode=False,
    paper_bgcolor='#1e1e1e',
    plot_bgcolor='#1e1e1e',
    font=dict(color='#d4d4d4'),
    xaxis_gridcolor='#333',
    yaxis_gridcolor='#333',
    xaxis2_gridcolor='#333',
    yaxis2_gridcolor='#333',
)

fig.show()